## **Installing Required Libraries**

In [4]:
!pip install requests

In [8]:
!pip install beautifulsoup4

## **Import Libraries**

In [13]:
import requests
from bs4 import BeautifulSoup
import csv
import pandas as pd
import time

## **Get URL and send Get Request**

In [19]:
url= "https://books.toscrape.com/"
response = requests.get(url)
if response.status_code==200:
    print("request successful")

else: 
    print("request failed")

request successful


## **Parse the HTM Content**

In [24]:
#Create a soup object to parse the HTML content
soup=BeautifulSoup(response.text, "html.parser")
print(soup) #this will enable you to see the html code of the site

<!DOCTYPE html>

<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-us"> <!--<![endif]-->
<head>
<title>
    All products | Books to Scrape - Sandbox
</title>
<meta content="text/html; charset=utf-8" http-equiv="content-type"/>
<meta content="24th Jun 2016 09:29" name="created"/>
<meta content="" name="description"/>
<meta content="width=device-width" name="viewport"/>
<meta content="NOARCHIVE,NOCACHE" name="robots"/>
<!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
<!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->
<link href="static/oscar/favicon.ico" rel="shortcut icon"/>
<link href="static/oscar/css/styles.css" rel="stylesheet" type="text/css"/>
<link href="s

## **Extract Details for Page 1**

In [29]:
#Find all 20 books on page 1
books = soup.find_all('h3')

start_time = time.time()
books_extracted = 0


#Iterate through the books and extract the information for each book
for book in books:
    book_url = book.find('a')['href']
    book_response = requests.get(url + book_url) #makes requests to the website as well as the book title info
    book_soup = BeautifulSoup(book_response.content, 'html.parser')

    #get title of the book
    title = book_soup.find("h1").text
    category = book_soup.find("ul",class_="breadcrumb").find_all('a')[2].text.strip() #you want to find all categories from where the list starts to end
    rating = book_soup.find('p', class_ ='star-rating')['class'][1]
    price = book_soup.find('p', class_='price_color').text.strip()
    availability = book_soup.find('p', class_= 'availability').text.strip()

    books_extracted += 1

    end_time= time.time()
    total_time = (end_time - start_time)/60.0

    print(f'Title:{title}')
    print(f'Category:{category}')
    print(f'Rating:{rating}')
    print(f'Price:{price}')
    print(f'Availability:{availability}')
    print('*******')


Title:A Light in the Attic
Category:Poetry
Rating:Three
Price:£51.77
Availability:In stock (22 available)
*******
Title:Tipping the Velvet
Category:Historical Fiction
Rating:One
Price:£53.74
Availability:In stock (20 available)
*******
Title:Soumission
Category:Fiction
Rating:One
Price:£50.10
Availability:In stock (20 available)
*******
Title:Sharp Objects
Category:Mystery
Rating:Four
Price:£47.82
Availability:In stock (20 available)
*******
Title:Sapiens: A Brief History of Humankind
Category:History
Rating:Five
Price:£54.23
Availability:In stock (20 available)
*******
Title:The Requiem Red
Category:Young Adult
Rating:One
Price:£22.65
Availability:In stock (19 available)
*******
Title:The Dirty Little Secrets of Getting Your Dream Job
Category:Business
Rating:Four
Price:£33.34
Availability:In stock (19 available)
*******
Title:The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull
Category:Default
Rating:Three
Price:£17.93
Availability:In stock (19 ava

## **Extract Details for all 50 Books**

In [37]:
books_data = []

start_time = time.time()
books_extracted = 0

for page_num in range(1, 51):
    page_url = f"https://books.toscrape.com/catalogue/page-{page_num}.html"
    response = requests.get(page_url)
    soup = BeautifulSoup(response.content, "html.parser")

    books = soup.find_all("article", class_="product_pod")

    for book in books:
        title = book.h3.a["title"]
        price = book.find("p", class_="price_color").text.strip()
        availability = book.find("p", class_="instock availability").text.strip()
        rating = book.find("p", class_="star-rating")["class"][1]

        books_data.append([title, rating, price, availability])
        books_extracted += 1

end_time = time.time()
total_time = (end_time - start_time) / 60

print(f"Books extracted: {books_extracted}")
print(f"Time taken: {total_time:.2f} minutes")

Books extracted: 1000
Time taken: 1.91 minutes


## **Export the Data**

In [40]:
#convert list to DataFrame
df = pd.DataFrame(books_data, columns=["Title", "Rating", "Price", "Availability"])
df.head()

,Title,Rating,Price,Availability
0,A Light in the Attic,Three,£51.77,In stock
1,Tipping the Velvet,One,£53.74,In stock
2,Soumission,One,£50.10,In stock
3,Sharp Objects,Four,£47.82,In stock
4,Sapiens: A Brief History of Humankind,Five,£54.23,In stock


In [42]:
#print first 10 rows
print(df.head(10))

                                               Title Rating   Price  \
0                               A Light in the Attic  Three  £51.77   
1                                 Tipping the Velvet    One  £53.74   
2                                         Soumission    One  £50.10   
3                                      Sharp Objects   Four  £47.82   
4              Sapiens: A Brief History of Humankind   Five  £54.23   
5                                    The Requiem Red    One  £22.65   
6  The Dirty Little Secrets of Getting Your Dream...   Four  £33.34   
7  The Coming Woman: A Novel Based on the Life of...  Three  £17.93   
8  The Boys in the Boat: Nine Americans and Their...   Four  £22.60   
9                                    The Black Maria    One  £52.15   

  Availability  
0     In stock  
1     In stock  
2     In stock  
3     In stock  
4     In stock  
5     In stock  
6     In stock  
7     In stock  
8     In stock  
9     In stock  


In [44]:
#Export to csv
df.to_csv("books_data.csv", index=False)